# IPL ML Models (Decision Tree vs Random Forest)

This notebook trains supervised learning models to predict player performance
based on venue and historical statistics.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [ ]:
deliveries = pd.read_csv("data/deliveries.csv")
matches = pd.read_csv("data/matches.csv")

print("Deliveries shape:", deliveries.shape)
print("Matches shape:", matches.shape)

deliveries.head()

In [ ]:
df = deliveries.merge(matches, left_on="match_id", right_on="id")

print("Merged df shape:", df.shape)
df.head()

In [ ]:
player_team_bat = df.groupby(["batter", "batting_team"]).size().reset_index(name="count")
player_team_bowl = df.groupby(["bowler", "bowling_team"]).size().reset_index(name="count")
bat_team = player_team_bat.sort_values("count", ascending=False).drop_duplicates("batter")
bat_team = bat_team[["batter", "batting_team"]]
bat_team.columns = ["player", "team"]

bowl_team = player_team_bowl.sort_values("count", ascending=False).drop_duplicates("bowler")
bowl_team = bowl_team[["bowler", "bowling_team"]]
bowl_team.columns = ["player", "team"]

In [ ]:
player_runs = df.groupby("batter")["batsman_runs"].sum().reset_index()
player_runs.columns = ["player", "total_runs"]

matches_played = df.groupby("batter")["match_id"].nunique().reset_index()
matches_played.columns = ["player", "matches"]


player_stats = player_runs.merge(matches_played, on="player")
player_stats["avg_runs"] = player_stats["total_runs"] / player_stats["matches"]
player_stats = player_stats.merge(player_team, on="player", how="left")

player_stats.head()

In [ ]:
wickets = df[df["dismissal_kind"].notnull()]
bowler_wickets = wickets.groupby("bowler").size().reset_index(name="wickets")

player_stats = player_stats.merge(
    bowler_wickets,
    left_on="player",
    right_on="bowler",
    how="left"
)

player_stats["wickets"] = player_stats["wickets"].fillna(0)
player_stats = player_stats.drop(columns=["bowler"])

player_stats.head()

In [ ]:
player_stats["performance"] = player_stats["total_runs"] + (player_stats["wickets"] * 20)

player_stats.head()

In [ ]:
def get_role(row):
    if row['wickets'] > 50 and row['total_runs'] > 1000:
        return "All-Rounder"
    elif row['wickets'] > 50:
        return "Bowler"
    else:
        return "Batsman"

player_stats["role"] = player_stats.apply(get_role, axis=1)
player_stats["role"].value_counts()

In [ ]:
# Venue-based player stats
venue_runs = df.groupby(["batter", "venue"])["batsman_runs"].sum().reset_index()
venue_runs.columns = ["player", "venue", "venue_runs"]

venue_matches = df.groupby(["batter", "venue"])["match_id"].nunique().reset_index()
venue_matches.columns = ["player", "venue", "venue_matches"]

venue_stats = venue_runs.merge(venue_matches, on=["player", "venue"])
venue_stats["venue_avg_runs"] = venue_stats["venue_runs"] / venue_stats["venue_matches"]

venue_stats.head()

In [ ]:
ml_df = venue_stats.merge(player_stats, on="player")

ml_df = ml_df[[
    "player",
    "venue",
    "venue_avg_runs",
    "avg_runs",
    "wickets",
    "performance",
    "role",
    "team"
]]

ml_df = ml_df.dropna()
ml_df.head()

In [ ]:
le_player = LabelEncoder()
le_venue = LabelEncoder()

ml_df["player_enc"] = le_player.fit_transform(ml_df["player"])
ml_df["venue_enc"] = le_venue.fit_transform(ml_df["venue"])

ml_df.head()

In [ ]:
X = ml_df[["player_enc", "venue_enc", "venue_avg_runs", "avg_runs", "wickets"]]
y = ml_df["performance"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

In [ ]:
#Train Decision Tree

dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

print("Decision Tree Results")
print("MAE:", mean_absolute_error(y_test, dt_pred))
print("MSE:", mean_squared_error(y_test, dt_pred))
print("R2 :", r2_score(y_test, dt_pred))

In [ ]:
#Train Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Results")
print("MAE:", mean_absolute_error(y_test, rf_pred))
print("MSE:", mean_squared_error(y_test, rf_pred))
print("R2 :", r2_score(y_test, rf_pred))

In [ ]:
rmse_dt = np.sqrt(mean_squared_error(y_test, dt_pred))
rmse_rf = np.sqrt(mean_squared_error(y_test, rf_pred))

print("Decision Tree RMSE:", rmse_dt)
print("Random Forest RMSE:", rmse_rf)

In [ ]:
results = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest"],
    "MAE": [
        mean_absolute_error(y_test, dt_pred),
        mean_absolute_error(y_test, rf_pred)
    ],
    "MSE": [
        mean_squared_error(y_test, dt_pred),
        mean_squared_error(y_test, rf_pred)
    ],
    "R2 Score": [
        r2_score(y_test, dt_pred),
        r2_score(y_test, rf_pred)
    ]
})

results

In [ ]:
best_model = results.sort_values(by="MAE").iloc[0]
print("Best Model Based on Lowest MAE:")
print(best_model)

In [ ]:
selected_team = "Mumbai Indians" #Chennai Super Kings
selected_venue = "Wankhede Stadium"
venue_data = ml_df[
    (ml_df["venue"] == selected_venue) &
    (ml_df["team"] == selected_team)
].copy()

venue_data.head()


In [ ]:
X_venue = venue_data[["player_enc", "venue_enc", "venue_avg_runs", "avg_runs", "wickets"]]

venue_data["dt_prediction"] = dt_model.predict(X_venue)
venue_data["rf_prediction"] = rf_model.predict(X_venue)

venue_data.head()

In [ ]:
dt_sorted = venue_data.sort_values(by="dt_prediction", ascending=False)

dt_xi = pd.concat([
    dt_sorted[dt_sorted["role"] == "Batsman"].head(5),
    dt_sorted[dt_sorted["role"] == "Bowler"].head(4),
    dt_sorted[dt_sorted["role"] == "All-Rounder"].head(2),
])

print("Decision Tree Best XI:")
dt_xi[["player", "role", "dt_prediction"]]

In [ ]:
rf_sorted = venue_data.sort_values(by="rf_prediction", ascending=False)

rf_xi = pd.concat([
    rf_sorted[rf_sorted["role"] == "Batsman"].head(5),
    rf_sorted[rf_sorted["role"] == "Bowler"].head(4),
    rf_sorted[rf_sorted["role"] == "All-Rounder"].head(2),
])

print("\nRandom Forest Best XI:")
rf_xi[["player", "role", "rf_prediction"]]